# Data Preparation for Predictive Analytics
This notebook merges the Medallion Architecture Gold Layer data (`fact_sales.csv`, `dim_date.csv`, `dim_product.csv`) into a flattened ML-ready dataset.

In [ ]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

# Define paths
base_path = '../Medallion Architecture/Gold Layer/'
fact_sales_path = os.path.join(base_path, 'fact_sales.csv')
dim_date_path = os.path.join(base_path, 'dim_date.csv')
dim_product_path = os.path.join(base_path, 'dim_product.csv')

## Load Gold Layer Data

In [ ]:
df_sales = pd.read_csv(fact_sales_path)
df_date = pd.read_csv(dim_date_path)
df_product = pd.read_csv(dim_product_path)

print(f"Sales shape: {df_sales.shape}")
print(f"Date shape: {df_date.shape}")
print(f"Product shape: {df_product.shape}")

## Merge Dimensions into Facts

In [ ]:
df_merged = df_sales.merge(df_date[['Date_Key', 'Date', 'Day', 'Month', 'Year', 'Is_Weekend']], on='Date_Key', how='left')
df_merged = df_merged.merge(df_product[['Product_Key', 'Parent ASIN', 'Title', 'Brand']], on='Product_Key', how='left')

# Convert Date to datetime
df_merged['Date'] = pd.to_datetime(df_merged['Date'])
df_merged = df_merged.sort_values('Date')
display(df_merged.head())

## Feature Engineering
Creating lag features and rolling averages for ML.

In [ ]:
# Calculate rolling 7-day averages for key metrics per product
df_merged['Revenue_7d_avg'] = df_merged.groupby('Product_Key')['Revenue'].transform(lambda x: x.rolling(7, min_periods=1).mean())
df_merged['Units_7d_avg'] = df_merged.groupby('Product_Key')['Units'].transform(lambda x: x.rolling(7, min_periods=1).mean())

# Flag Bleeding SKUs (Net Margin < 0.1)
df_merged['Is_Bleeding'] = (df_merged['Net Margin'] < 0.10).astype(int)

df_merged.to_csv('ml_ready_data.csv', index=False)
print("Saved ML-ready data to 'ml_ready_data.csv'")